## Lawrence street lawrence MA postfix

In [2]:
from pathlib import Path
import pandas as pd
import re

CACHE_CSV = Path("/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/geocode_cache_t.csv")

cache = pd.read_csv(CACHE_CSV)

# Backup first
backup_path = CACHE_CSV.with_name("geocode_cache_backup_before_lawrence_fix.csv")
cache.to_csv(backup_path, index=False)
print(f"Backup saved to: {backup_path}")

def fix_lawrence_city_bug(cleaned_address):
    if pd.isna(cleaned_address):
        return cleaned_address

    addr = str(cleaned_address).strip()

    # Only target rows ending in ", MA" but NOT containing ", Lawrence, MA"
    # Example: "1905 LAWRENCE ST, MA"
    if re.search(r",\s*MA$", addr, flags=re.IGNORECASE) and not re.search(
        r",\s*LAWRENCE\s*,?\s*MA$", addr, flags=re.IGNORECASE
    ):
        addr = re.sub(r",\s*MA$", "", addr, flags=re.IGNORECASE).strip()
        addr = f"{addr}, Lawrence, MA"

    return addr

mask = (
    cache["cleaned_address"].astype(str).str.contains(r",\s*MA$", case=False, regex=True)
    & ~cache["cleaned_address"].astype(str).str.contains(r",\s*LAWRENCE\s*,?\s*MA$", case=False, regex=True)
)

print(f"Rows to fix: {mask.sum()}")

cache.loc[mask, "cleaned_address"] = cache.loc[mask, "cleaned_address"].apply(fix_lawrence_city_bug)

cache.to_csv(CACHE_CSV, index=False)

print(f"Fixed cache saved to: {CACHE_CSV}")

Backup saved to: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/geocode_cache_backup_before_lawrence_fix.csv
Rows to fix: 0
Fixed cache saved to: /Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/geocode_cache_t.csv


## Checks if the changed coordinates are good

In [4]:
import pandas as pd
import re

cache = pd.read_csv("/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/geocode_cache_t.csv")

# Rows affected by the Lawrence Street bug:
# Example:
#   "1905 LAWRENCE ST, MA"
#   "139 PARK ST & LAWRENCE ST, MA"
#
# but NOT:
#   "..., Lawrence, MA"

affected_mask = (
    cache["cleaned_address"]
    .astype(str)
    .str.contains(r",\s*MA$", case=False, regex=True)
    &
    ~cache["cleaned_address"]
    .astype(str)
    .str.contains(r",\s*LAWRENCE\s*,?\s*MA$", case=False, regex=True)
)

affected = cache.loc[affected_mask].copy()

print(f"Affected rows: {len(affected):,}")

# Basic coordinate sanity checks

affected["coord_valid"] = (
    affected["lat"].between(40, 48)
    &
    affected["long"].between(-74, -66)
)

# Lawrence is roughly:
# Lat: 42.65 - 42.75
# Lon: -71.25 - -71.10

affected["near_lawrence"] = (
    affected["lat"].between(42.60, 42.80)
    &
    affected["long"].between(-71.35, -71.00)
)

print("\nRows outside New England bounds:")
display(
    affected.loc[
        ~affected["coord_valid"],
        ["raw_address", "cleaned_address", "lat", "long"]
    ]
)

print("\nRows not near Lawrence:")
display(
    affected.loc[
        ~affected["near_lawrence"],
        ["raw_address", "cleaned_address", "lat", "long"]
    ].sort_values("lat")
)

print("\nSummary:")
print(f"Total affected rows: {len(affected)}")
print(f"Outside MA/NE bounds: {(~affected['coord_valid']).sum()}")
print(f"Not near Lawrence: {(~affected['near_lawrence']).sum()}")

Affected rows: 0

Rows outside New England bounds:


,raw_address,cleaned_address,lat,long



Rows not near Lawrence:


,raw_address,cleaned_address,lat,long



Summary:
Total affected rows: 0
Outside MA/NE bounds: 0
Not near Lawrence: 0


## Clean fl, apt, "'", and "#"

In [6]:
import pandas as pd
import re

CACHE_CSV = "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/geocode_cache_t.csv"

cache = pd.read_csv(CACHE_CSV)

def clean_for_dedupe(addr):
    if pd.isna(addr):
        return ""

    addr = str(addr).upper()

    # Remove apostrophes
    addr = addr.replace("'", "")

    # Remove apartment/floor/unit info
    addr = re.sub(
        r"\s+(?:FL|FLOOR|APT|APARTMENT|UNIT)\b.*$",
        "",
        addr,
        flags=re.IGNORECASE,
    )

    # Remove # info
    addr = re.sub(r"\s+#.*$", "", addr)

    # Normalize whitespace
    addr = re.sub(r"\s+", " ", addr).strip()

    return addr

cache["dedupe_address"] = cache["raw_address"].apply(clean_for_dedupe)

print(f"Original rows: {len(cache):,}")
print(f"Unique raw addresses: {cache['raw_address'].nunique():,}")
print(f"Unique cleaned addresses: {cache['dedupe_address'].nunique():,}")

print(
    f"Addresses merged by cleaning: "
    f"{cache['raw_address'].nunique() - cache['dedupe_address'].nunique():,}"
)

# Show examples that collapsed together
duplicates = (
    cache.groupby("dedupe_address")["raw_address"]
    .agg(list)
    .reset_index()
)

duplicates = duplicates[
    duplicates["raw_address"].apply(len) > 1
]

print(f"\nCollapsed address groups: {len(duplicates):,}")

display(
    duplicates.head(50)
)

Original rows: 58,265
Unique raw addresses: 58,265
Unique cleaned addresses: 33,222
Addresses merged by cleaning: 25,043

Collapsed address groups: 7,925


,dedupe_address,raw_address
10,0 BENNINGTON ST,"[0 BENNINGTON ST, 0 BENNINGTON ST #3, 0 BENNIN..."
57,02 MUSEUM SQ,"[02 MUSEUM SQ #711, 02 MUSEUM SQ #803, 02 MUSE..."
75,04 INMAN ST,"[04 INMAN ST #11, 04 INMAN ST FL 2]"
127,09 SUMMER ST,"[09 SUMMER ST #709, 09 SUMMER ST #S708]"
133,1 ALBION ST,"[1 ALBION ST, 1 ALBION ST FL 1]"
146,1 BAILEY ST,"[1 BAILEY ST, 1 BAILEY ST FL 1]"
147,1 BALLARD WY,"[1 BALLARD WY, 1 BALLARD WY #101, 1 BALLARD WY..."
153,1 BEACON AV,"[1 BEACON AV, 1 BEACON AV #1 FL 1, 1 BEACON AV..."
154,1 BEACON AVE,"[1 BEACON AVE, 1 BEACON AVE #236]"
155,1 BEACON ST,"[1 BEACON ST, 1 BEACON ST #303, 1 BEACON ST #409]"


## Changes the csv

In [8]:
deduped = (
    cache
    .sort_values("address_confidence", ascending=False)
    .drop_duplicates(subset="dedupe_address", keep="first")
)

deduped.to_csv(
    "/Users/sreej/Desktop/Gateway_lawrence/gatewayinitiative-lawrencepd/scripts/data/missing_dates_csv/md_checkpoints/geocode_cache_deduped.csv",
    index=False
)

print(f"Deduped rows: {len(deduped):,}")

Deduped rows: 33,222
